In [1]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_opt import logging as ls_opt_logging
from cardiac_electrophysiology.ls_opt import optimizer
from cardiac_electrophysiology.utils import analysis, visualization

In [2]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.005,
        tau=100,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

optimizer_settings = optimizer.LBFGSConfig(
    maximum_num_iterations=10,
    relative_function_tolerance= 1e-6,
    relative_gradient_tolerance=1e-6,
    max_line_search_steps=100,
)
ls_opt_logger_settings = ls_opt_logging.LSOPTLoggerSettings(
    print_to_console=True,
    logfile_path= None,
)

In [3]:
posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

Widget(value='<iframe src="http://localhost:34741/index.html?ui=P_0x7f1ddf937cb0_0&reconnect=auto" class="pyvi…

In [6]:
initial_guess = np.zeros_like(additional_output.prior_mean_parameter)
ls_optimizer = optimizer.LBFGSOptimizer(optimizer_settings, ls_opt_logger_settings)
map_result = ls_optimizer.run(
    initial_guess=initial_guess,
    loss_function=posterior.evaluate_cost,
    gradient_function=posterior.evaluate_gradient,
)
# np.save("../results/map_estimate.npy", map_result.result)

| Iteration   | Time        | Loss        | Grad Norm   | 
---------------------------------------------------------
| +1.000e+00  | +2.723e+00  | +2.935e+07  | +1.191e+06  | 
| +2.000e+00  | +4.047e+00  | +2.294e+07  | +9.292e+06  | 
| +3.000e+00  | +5.376e+00  | +2.101e+07  | +8.008e+06  | 
| +4.000e+00  | +6.720e+00  | +2.098e+07  | +2.307e+06  | 
| +5.000e+00  | +7.996e+00  | +2.064e+07  | +1.824e+06  | 
| +6.000e+00  | +9.269e+00  | +1.938e+07  | +2.733e+06  | 
| +7.000e+00  | +1.057e+01  | +1.805e+07  | +3.668e+06  | 
| +8.000e+00  | +1.192e+01  | +1.630e+07  | +2.698e+06  | 
| +9.000e+00  | +1.324e+01  | +1.563e+07  | +1.913e+06  | 
| +1.000e+01  | +1.452e+01  | +1.511e+07  | +2.189e+06  | 


In [ ]:
map_parameter = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

In [ ]:
for data in (
    analysis_data.prior_mean_parameter,
    analysis_data.ground_truth_parameter,
    analysis_data.map_parameter,
    analysis_data.diff_lat_truth_prior,
    analysis_data.diff_lat_truth_map,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
    )